In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader as pdr
import statsmodels.api as sm

startdate = '1980-01-01'

# 25 size/BM portfolios
Rets = pdr.DataReader('25_Portfolios_5x5', 'famafrench', start=startdate)[0]

# Fama-French factors and RF
FF = pdr.DataReader('F-F_Research_Data_Factors', 'famafrench', start=startdate)[0]

# excess returns
Rets = Rets.apply(lambda x: x - FF['RF'])

## Factor Analysis

Sklearn's `FactorAnalysis` estimates the loading matrix $B$ and idiosyncratic variance matrix $D$ by maximum likelihood, given the model $\text{Cov}(R) = BB' + D$ where $D$ is diagonal.  It then infers factors at each date by GLS:
$$f_t = (B'D^{-1}B)^{-1}B'D^{-1}(R_t - \hat{\mu})$$

Key attributes:
- `components_` = $B'$ (K×N loading matrix)
- `noise_variance_` = diagonal of $D$ (idiosyncratic variances)
- `transform()` / `fit_transform()` = inferred factors

In [ ]:
from sklearn.decomposition import FactorAnalysis

fa = FactorAnalysis(n_components=3).fit(Rets)
print('components_ shape:', fa.components_.shape)
print('\nnoise_variance_:\n', fa.noise_variance_.round(2))

fa_factors = pd.DataFrame(
    fa.transform(Rets), index=Rets.index, columns=['FA1', 'FA2', 'FA3']
)

## Principal Components

Sklearn's `PCA` extracts the eigenvectors of the covariance matrix corresponding to the largest eigenvalues.  The factors are projections of the demeaned returns onto these eigenvectors: $f_t = (C_K'C_K)^{-1}C_K'(R_t - \hat{\mu}) = C_K'(R_t - \hat{\mu})$.

Key attributes:
- `components_` = $C_K'$ (K×N, the transposed eigenvectors)
- `explained_variance_ratio_` = fraction of total variance explained per component
- `transform()` / `fit_transform()` = the factors $f_t$

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3).fit(Rets)
print('Variance explained:', pca.explained_variance_ratio_.round(4))
print('Cumulative:        ', pca.explained_variance_ratio_.cumsum().round(4))

pca_factors = pd.DataFrame(
    pca.transform(Rets), index=Rets.index, columns=['PC1', 'PC2', 'PC3']
)

## Factor-Mimicking Portfolios

The raw factors from both methods are linear combinations of returns, but the weights don't sum to 1.  We normalize to get traded portfolios (factor-mimicking portfolios, or FMPs).

- **FA:** The GLS weight matrix is $W = (B'D^{-1}B)^{-1}B'D^{-1}$, computed directly from `components_` and `noise_variance_`.  Each column gives a factor's weights on the $N$ returns; we normalize each to sum to 1.
- **PCA:** The eigenvector entries are the weights; we divide each by its sum.

In [ ]:
# FA FMPs: compute GLS weights directly from sklearn attributes
B = fa.components_.T                          # N×K
D_inv = np.diag(1 / fa.noise_variance_)       # N×N
W = np.linalg.solve(B.T @ D_inv @ B, B.T @ D_inv).T  # N×K
W_norm = W / W.sum(axis=0)
fa_fmp = pd.DataFrame(Rets.to_numpy() @ W_norm, index=Rets.index, columns=['FA1', 'FA2', 'FA3'])

# PCA FMPs: normalize eigenvectors
C = pca.components_.T                         # N×K
C_norm = C / C.sum(axis=0)
pca_fmp = pd.DataFrame(Rets.to_numpy() @ C_norm, index=Rets.index, columns=['PC1', 'PC2', 'PC3'])

## Sharpe Ratio Comparison

The punchline: all three sets of factors (FA, PCA, Fama-French) deliver similar maximum Sharpe ratios.  The maximum Sharpe ratio from a set of factors is
$$\sqrt{\mu'\Sigma^{-1}\mu}$$
where $\mu$ and $\Sigma$ are the mean vector and covariance matrix of the factor-mimicking portfolio excess returns.

In [ ]:
def max_sharpe(df):
    mu = df.mean().values
    Sigma = df.cov().values
    return np.sqrt(mu @ np.linalg.inv(Sigma) @ mu)

results = pd.Series({
    'Fama-French': max_sharpe(FF[['Mkt-RF', 'SMB', 'HML']]),
    'Factor Analysis': max_sharpe(fa_fmp),
    'PCA': max_sharpe(pca_fmp),
})

print('Monthly Max Sharpe Ratios')
print('=' * 30)
print(results.round(4).to_string())

In [ ]:
import matplotlib.pyplot as plt

pca_full = PCA().fit(Rets)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, 11), pca_full.explained_variance_ratio_[:10])
ax.set_xlabel('Component')
ax.set_ylabel('Fraction of Variance')
ax.set_title('PCA Scree Plot (first 10 components)')
plt.tight_layout()
plt.show()

## Scree Plot

Each eigenvalue equals the variance of the corresponding principal component.  The scree plot shows the fraction of total variance explained by each of the first 10 components.  The first component dominates, capturing about 82% of variance.